In [1]:
%pip install transformers datasets torch scikit-learn

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
from sklearn.metrics import accuracy_score
import numpy as np

# =========================================================
# STEP 0: Create a dataset
# =========================================================

# 3 classes:
# 0 = negative
# 1 = neutral
# 2 = positive

texts = [
    "I hated this movie",
    "The food was terrible",
    "This product is awful",
    "Nothing special about the service",
    "The weather is average today",
    "It works as expected",
    "I absolutely loved it",
    "Amazing experience overall",
    "This is the best purchase ever",
    
    "Bad customer support",
    "The app crashes often",
    "Very disappointing quality",
    "The package arrived",
    "The meeting was okay",
    "It is a standard laptop",
    "Fantastic performance",
    "Really happy with the results",
    "Excellent design and usability",
]

labels = [
    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
    
    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
]

# =========================================================
# STEP 1: Tokenization
# =========================================================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=64
)

dataset_dict = {
    "input_ids": encodings["input_ids"],
    "attention_mask": encodings["attention_mask"],
    "labels": labels
}

dataset = Dataset.from_dict(dataset_dict)

# Split dataset into train/validation/test
train_test = dataset.train_test_split(test_size=0.3, seed=42)
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)

train_data = train_test["train"]
val_data = test_valid["train"]
test_data = test_valid["test"]

# =========================================================
# STEP 2: Load pre-trained BERT model
# =========================================================

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Freeze all layers
for param in model.base_model.parameters():
    param.requires_grad = False

# Unfreeze last 2 encoder layers
for param in model.base_model.encoder.layer[-2:].parameters():
    param.requires_grad = True

# =========================================================
# STEP 3: Metrics
# =========================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# =========================================================
# STEP 4: Training arguments
# =========================================================

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

# =========================================================
# STEP 5: Trainer
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
)

# =========================================================
# STEP 6: Train
# =========================================================

trainer.train()

# =========================================================
# STEP 7: Evaluate
# =========================================================

results = trainer.evaluate(eval_dataset=test_data)

print(f"Test Accuracy: {results['eval_accuracy']:.4f}")


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3576.30it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSI

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.014200,0.333333
2,No log,1.022306,0.333333
3,No log,1.026552,0.333333


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.55s/it]
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.86s/it]
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.59s/it]
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,1.069975,3,0.666667


Test Accuracy: 0.6667
